<a href="https://colab.research.google.com/github/Antibote/pinn-eos-aluminum/blob/main/notebooks/section3_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Раздел 3. Учёт физических ограничений (PINN)

Порядок: **раздел 0** → 3.1 → 3.2 → 3.3 → 3.4. `split.npz` и `normalization.json` уже лежат в `artifacts/` репозитория.

- **3.1** — физически ориентированная функция потерь: MSE по данным + штрафы (∂P/∂ρ)_T > 0, (∂e/∂T)_ρ ≥ 0 и 0 < c_s < c;
- **3.2** — обучение PINN (256×4, GELU, обычная сумма градиентов);
- **3.3** — ограничение на скорость звука c_s² = (∂P/∂ρ)_T + T (∂P/∂T)²_ρ / (ρ² (∂e/∂T)_ρ), 0 < c_s < c;
- **3.4** — сравнение с базовой моделью по точности и нарушениям физики.

Производные — autograd по ln ρ и ln T. Код — `src/eos/pinn.py`. Результаты сохраняются в `artifacts/` и отправляются на GitHub через `save_to_repo`.

## 0. Подготовка (общая, запускать каждому)

In [1]:
import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите GPU в настройках среды')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch 2.11.0+cu130
GPU: Tesla T4


In [ ]:
# Репозиторий: код (src/eos) и папка artifacts — сюда сохраняются все результаты.
# На GitHub их отправляет save_to_repo('что сохранили') — см. ячейку ниже.
import json, subprocess, sys, urllib.parse, urllib.request, zipfile
from pathlib import Path

REPO = Path('pinn-eos-aluminum')
REPO_URL = 'https://github.com/Antibote/pinn-eos-aluminum.git'
_TOKEN = None


def git(*args):
    """git в папке репозитория. Токен в сообщениях об ошибке скрыт."""
    r = subprocess.run(['git', '-C', str(REPO), *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        raise RuntimeError(f'git {args[0]}: ' + (msg.replace(_TOKEN, '***') if _TOKEN else msg))
    return r.stdout


if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
git('config', 'user.name', 'Colab')
git('config', 'user.email', 'colab@users.noreply.github.com')
git('pull', '-q', '--rebase', '--autostash')
sys.path.insert(0, str(REPO / 'src'))

ARTIFACTS = REPO / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Артефакты:', ARTIFACTS.resolve())


In [ ]:
def save_to_repo(message):
    """Коммитит папку artifacts и отправляет её на GitHub.

    Нужен секрет Colab GITHUB_TOKEN (значок ключа слева) с доступом к репозиторию.

    Args:
        message: что сохранили, например 'Бейзлайн: метрики и карта ошибок'.
    """
    global _TOKEN
    from google.colab import userdata
    _TOKEN = userdata.get('GITHUB_TOKEN')

    big = [str(p) for p in ARTIFACTS.rglob('*') if p.is_file() and p.stat().st_size > 50 * 2**20]
    if big:
        raise RuntimeError(f'Файлы больше 50 МБ GitHub не примет: {big}')
    git('add', 'artifacts')
    changed = git('diff', '--cached', '--name-only', '--', 'artifacts').split()
    if not changed:
        print('Нечего сохранять: artifacts не изменилась')
        return

    # Автор коммита — владелец токена
    req = urllib.request.Request('https://api.github.com/user', headers={'Authorization': f'token {_TOKEN}'})
    me = json.load(urllib.request.urlopen(req))
    author = f"{me.get('name') or me['login']} <{me['id']}+{me['login']}@users.noreply.github.com>"

    git('commit', '-q', '-m', message, '--author', author)
    url = REPO_URL.replace('https://', f'https://x-access-token:{_TOKEN}@')
    git('pull', '-q', '--rebase', url, 'main')
    git('push', '-q', url, 'HEAD:main')
    print('На GitHub:', git('log', '-1', '--format=%h %s').strip())
    print('Файлы:', *changed, sep='\n  ')


In [ ]:
# Данные с Яндекс Диска (в репозиторий не попадают)
DATA = REPO / 'data'
if not list(DATA.glob('Obl*.txt')):
    api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?' + urllib.parse.urlencode(
        {'public_key': 'https://disk.yandex.ru/d/iUSbYrJcsbXSbw', 'path': '/Материалы к хакатону.zip'})
    href = json.load(urllib.request.urlopen(api))['href']
    urllib.request.urlretrieve(href, 'data.zip')
    zipfile.ZipFile('data.zip').extractall(DATA)
print(len(list(DATA.glob('Obl*.txt'))), 'файлов данных в', DATA)


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eos.data import load_all, region_grid

df = load_all(DATA)
print(df.shape)
df.head()

(1130053, 5)


,rho,T,P,E,region
0,0.001000,0.1,0.000515,-46.797895,Obl01
1,0.001030,0.1,0.000540,-46.790300,Obl01
2,0.001060,0.1,0.000569,-46.782552,Obl01
3,0.001091,0.1,0.000597,-46.774662,Obl01
4,0.001124,0.1,0.000626,-46.766630,Obl01


---
## 3. Учёт физических ограничений

### 3.1 Физически ориентированная функция потерь

In [ ]:
# 3.1 Физически ориентированная функция потерь (код — src/eos/pinn.py)
import importlib, json, torch
import eos.normalize as nz, eos.model as em, eos.pinn as pn, eos.metrics as mt, eos.baseline as bl, eos.inference as inf
for m in (nz, em, mt, pn, bl, inf):
    importlib.reload(m)
pd.set_option('display.width', 250)
print('Устройство:', DEVICE)   # должно быть cuda — иначе включите GPU в настройках среды

norm = nz.EOSNormalizer.load(ARTIFACTS / 'normalization.json')
split = dict(np.load(ARTIFACTS / 'split.npz'))
test_df = df.iloc[split['test']]

def to_t(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)

trainer = pn.Trainer(*(to_t(df[c].values) for c in ['rho', 'T', 'P', 'e']),
                     idx_train=torch.tensor(split['train'], device=DEVICE),
                     idx_val=torch.tensor(split['val'], device=DEVICE),
                     w_train=to_t(em.region_weights(df.iloc[split['train']])))

# Функция потерь PINN (производные — autograd по ln ρ и ln T, штрафы безразмерные):
#   L = MSE(данные)                                     — log10 P и asinh e после RobustScaler
#     + λ_ρ · mean max(0, −∂lnP/∂lnρ)                  — механическая устойчивость (∂P/∂ρ)_T > 0
#     + λ_T · mean max(0, −∂e/∂lnT) / (|e| + 1)         — термическая устойчивость (∂e/∂T)_ρ ≥ 0
#     + λ_c · mean нарушения 0 < c_s < c                 — скорость звука (3.3)
# Штрафы считаются в n_coll случайных точках по всей области (log ρ, log T); λ растут первые 20% шагов.
# Сеть 256×4, активация GELU (гладкая — производные сети входят в loss), градиенты — обычная сумма.
CFG = dict(hidden=256, layers=4, activation='gelu', loss='mse', grad='fixed',
           lam={'mono_rho': 1, 'mono_T': 1, 'sound': 1}, batch=8192, n_coll=1024, lr=2e-3)
STEPS = 30_000   # для пробного запуска поставьте 2000

# Штрафы на необученной сети (0 — условие выполнено во всех точках коллокации)
probe = em.EOSNet(norm, hidden=CFG['hidden'], layers=CFG['layers'], activation=CFG['activation']).to(DEVICE)
print('Штрафы на необученной сети:', {k: round(v.item(), 5) for k, v in trainer.physics_loss(probe, 4096).items()})

# Термодинамическая согласованность (∂e/∂ρ)_T = (P − T ∂P/∂T)/ρ² в loss не входит: её нарушает
# сама таблица (ratio_median = 0.5 в холодной части вместо 1), штраф спорил бы с данными
cons_table = pn.table_consistency(df)
cons_table.to_csv(ARTIFACTS / 'table_consistency.csv', index=False)
print(cons_table.to_string(index=False, float_format=lambda v: f'{v:.3g}'))

### 3.2 Обучение PINN

In [ ]:
# 3.2 Обучение PINN (прогресс печатается каждые 1000 шагов)
torch.manual_seed(0)
pinn = em.EOSNet(norm, hidden=CFG['hidden'], layers=CFG['layers'], activation=CFG['activation']).to(DEVICE)
hist = trainer.fit(pinn, lam=CFG['lam'], steps=STEPS, loss=CFG['loss'], grad=CFG['grad'],
                   batch=CFG['batch'], n_coll=CFG['n_coll'], lr=CFG['lr'])
print(f"Обучение: {hist['time_s'] / 60:.1f} мин, лучший val MSE {hist['best_val']:.3e}")
torch.save({'state_dict': pinn.state_dict(), 'config': pinn.config, 'train': CFG, 'steps': STEPS},
           ARTIFACTS / 'pinn.pt')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(hist['step'], hist['val']); ax[0].set(yscale='log', xlabel='шаг', ylabel='MSE по данным (val)', title='Точность')
ax[1].plot(hist['step'], hist['phys']); ax[1].set(yscale='symlog', xlabel='шаг', ylabel='λ · штраф (train)', title='Физика')
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_learning_curve.png', dpi=130, bbox_inches='tight')
plt.show()
save_to_repo('3.2: PINN (GELU, MSE, штрафы ∂P/∂ρ, ∂e/∂T, c_s) — веса и кривая обучения')

### 3.3 Ограничение на скорость звука 0 < c_s < c

In [ ]:
# 3.3 Ограничение на скорость звука 0 < c_s < c (c — скорость света)
#   c_s² = (∂P/∂ρ)_T + T (∂P/∂T)_ρ² / (ρ² (∂e/∂T)_ρ);  в кДж/г = (км/с)²
# У PINN производные точные (autograd), у таблицы и бейзлайна — конечные разности
from eos.metrics import C_LIGHT_KMS
model_pinn = inf.EOSModel.from_net(pinn, device=DEVICE)

# Доля нарушений в 100 000 случайных точках всей области
rho_q, T_q = bl.random_queries(100_000, seed=1)
d = model_pinn._log_derivatives(rho_q, T_q)[0]
cs2_pinn = d['cs2']
baseline_full = bl.BilinearEOS(df)
viol_base = mt.physics_violations(baseline_full.predict, rho_q, T_q)['viol_cs_%']
print(f"Нарушение 0 < c_s < c: PINN {100 * np.mean(~((cs2_pinn > 0) & (cs2_pinn < C_LIGHT_KMS ** 2))):.4f}%, "
      f"бейзлайн (таблица) {viol_base:.4f}%")

# Карта c_s на плоскости (ρ, T) и срезы c_s(ρ) при фиксированных T
lr, lt = np.meshgrid(np.linspace(-3, 4, 300), np.linspace(-1, 7, 300))
cs_map = model_pinn.sound_speed(10 ** lr, 10 ** lt)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
im = ax[0].pcolormesh(lr, lt, np.log10(cs_map), cmap='viridis', shading='auto')
plt.colorbar(im, ax=ax[0], label='log10 c_s, км/с')
ax[0].set(xlabel='log10 ρ, г/см³', ylabel='log10 T, K', title='PINN: скорость звука c_s (белое — c_s² ≤ 0)')
rho_line = np.logspace(-3, 4, 600)
for k, Tv in enumerate([10.0, 1e3, 1e5, 1e7]):
    ax[1].plot(rho_line, model_pinn.sound_speed(rho_line, np.full_like(rho_line, Tv)), color=f'C{k}', label=f'PINN, T={Tv:g} K')
ax[1].axhline(C_LIGHT_KMS, color='r', ls='--', label='скорость света c')
ax[1].set(xscale='log', yscale='log', xlabel='ρ, г/см³', ylabel='c_s, км/с', title='c_s(ρ): 0 < c_s < c')
ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_sound_speed.png', dpi=130, bbox_inches='tight')
plt.show()

### 3.4 Сравнение с базовой моделью

In [ ]:
# 3.4 Сравнение с базовой моделью: точность и нарушения физики на одном test (функция mt.evaluate)


class TorchEOS:
    """Обёртка для замера скорости: float32 на DEVICE, как в рабочем инференсе."""

    def __init__(self, model):
        self.model = model.eval()

    @torch.no_grad()
    def predict(self, rho, T):
        P, e = self.model(to_t(rho), to_t(T))
        return P.cpu().numpy(), e.cpu().numpy()


metrics, per_point = mt.evaluate(pn.numpy_predict(pinn), test_df, 'PINN')
metrics['time_1e5_s'] = bl.benchmark(TorchEOS(pinn), n=100_000)
metrics['size_MB'] = sum(p.numel() for p in pinn.parameters()) * 4 / 2**20
metrics.update({f'autograd_{k}': v for k, v in pn.physics_report(pinn, n=100_000).items()})
with open(ARTIFACTS / 'metrics_pinn.json', 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

# Бейзлайн пересчитываем (быстро), чтобы в его метриках было и нарушение скорости звука
with open(ARTIFACTS / 'metrics_baseline.json') as f:
    base_old = json.load(f)
base_met, _ = mt.evaluate(bl.BilinearEOS(df, decimate=True).predict, test_df, 'Baseline (билинейная)')
base_met.update({k: base_old[k] for k in ('time_1e5_s', 'size_MB')})
rows = [base_met]
if (ARTIFACTS / 'metrics_vanilla.json').exists():          # Vanilla — из раздела 2 (без viol_cs_%)
    with open(ARTIFACTS / 'metrics_vanilla.json') as f:
        rows.append(json.load(f))
rows.append(metrics)
keys = ['err_P_median_%', 'err_P_p99_%', 'err_P_max_%', 'err_e_median_%', 'err_e_p99_%', 'err_e_max_%',
        'viol_dP_drho_%', 'viol_de_dT_%', 'viol_cs_%', 'viol_dP_dT_%', 'time_1e5_s', 'size_MB']
compare = pd.DataFrame(rows).set_index('model')
compare.to_csv(ARTIFACTS / 'metrics_compare.csv')
display(compare[keys].T.round(4))
print('Физика PINN через autograd (100 000 случайных точек):')
print(pd.Series({k: v for k, v in metrics.items() if k.startswith('autograd_')}).round(4).to_string())

# Карта ошибок на (ρ, T)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, col, title in [(ax[0], 'err_P', 'ошибка P, %'), (ax[1], 'err_e', 'ошибка e, %')]:
    sc = a.scatter(per_point['rho'], per_point['T'], c=np.log10(per_point[col] * 100 + 1e-6), s=0.3,
                   cmap='viridis', vmin=-3, vmax=2, rasterized=True)
    a.set(xscale='log', yscale='log', xlabel='ρ, г/см³', ylabel='T, K', title=f'PINN: {title}')
    plt.colorbar(sc, ax=a, label='log10(ошибка, %)')
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_errors.png', dpi=130, bbox_inches='tight')
plt.show()


def section(col, v):
    """Точки таблицы при col ≈ v: в каждой области свой ближайший узел (сетки разные)."""
    parts = []
    for _, g in df.groupby('region'):
        u = np.unique(g[col])
        k = np.abs(np.log(u / v)).argmin()
        if abs(np.log(u[k] / v)) < 0.02:
            parts.append(g[g[col] == u[k]])
    return pd.concat(parts)


# Сечения: таблица (точки) и PINN (линии); внизу — знак производных
predict = pn.numpy_predict(pinn)
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
T_line, rho_line = np.logspace(-1, 7, 600), np.logspace(-3, 4, 600)
for k, rv in enumerate([0.01, 2.7, 100.0]):
    pts, r_ = section('rho', rv), np.full_like(T_line, rv)
    ax[0, 0].scatter(pts['T'], pts['e'], s=3, color=f'C{k}', alpha=0.3)
    ax[0, 0].plot(T_line, predict(r_, T_line)[1], color=f'C{k}', lw=1, label=f'ρ={rv:g}')
    d = pn.log_derivatives(pinn, to_t(r_), to_t(T_line), create_graph=False)
    ax[1, 0].plot(T_line, d['e_t'].detach().cpu().numpy(), color=f'C{k}', lw=1, label=f'ρ={rv:g}')
for k, Tv in enumerate([10.0, 1e3, 1e5]):
    pts, T_ = section('T', Tv), np.full_like(rho_line, Tv)
    ax[0, 1].scatter(pts['rho'], pts['P'], s=3, color=f'C{k}', alpha=0.3)
    ax[0, 1].plot(rho_line, predict(rho_line, T_)[0], color=f'C{k}', lw=1, label=f'T={Tv:g} K')
    d = pn.log_derivatives(pinn, to_t(rho_line), to_t(T_), create_graph=False)
    ax[1, 1].plot(rho_line, d['a'].detach().cpu().numpy(), color=f'C{k}', lw=1, label=f'T={Tv:g} K')
ax[0, 0].set(xscale='log', yscale='symlog', xlabel='T, K', ylabel='e, кДж/г', title='e(T): точки — таблица, линии — PINN')
ax[0, 1].set(xscale='log', yscale='log', xlabel='ρ, г/см³', ylabel='P, ГПа', title='P(ρ): точки — таблица, линии — PINN')
ax[1, 0].set(xscale='log', yscale='symlog', xlabel='T, K', ylabel='∂e/∂lnT', title='∂e/∂lnT ≥ 0 — термическая устойчивость')
ax[1, 1].set(xscale='log', xlabel='ρ, г/см³', ylabel='∂lnP/∂lnρ', title='∂lnP/∂lnρ > 0 — механическая устойчивость')
for a in ax.ravel():
    a.grid(alpha=0.3); a.legend(fontsize=8)
ax[1, 0].axhline(0, color='k', lw=0.8); ax[1, 1].axhline(0, color='k', lw=0.8)
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_sections.png', dpi=130, bbox_inches='tight')
plt.show()

save_to_repo('3.3–3.4: PINN — скорость звука, метрики на test, карта ошибок, сечения')